<a href="https://colab.research.google.com/github/choaib47/choaib47/blob/main/M323_Woche2_Dokumentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Dokumentation M323 – Woche 2**
Inhalte: **repetition (pure functions / side effects), abstraktion, closures, higher-order functions wie `map()` und `reduce()`**.


## Lernziele Woche 2
- repetition der vorwoche
- besprechung der übung
- **abstraktion** (werte in variablen / funktionen kapseln)
- **higher-order functions** und **closures**
- einsatz von **`map()`** und **`reduce()`** (und verwandte funktionen)


# Abstraktion
abstraktion bedeutet hier: werte (und später auch verhalten) so zu kapseln, dass code **lesbarer**, **wiederverwendbarer** und **weniger fehleranfällig** wird.

Beispiel:

In [ ]:
# 323 - Beispiel-1: Abstraktion von Werten in Variablen
#
gewuerz = ["Pfeffer", "Salz", "Anis", "Curry", "Safran"]

print("Gewürze1:")
print(gewuerz[0])
print(gewuerz[1])

gewuerz.append("Koreander")
def showSpice():
	for i in gewuerz:
		print(i)

print()
print("Gewürze2:")
showSpice()


# Repetition: Pure vs. Impure Functions
das beispiel zeigt typische probleme bei **impure** code:


In [ ]:
# Beispiel: Pure Functions

currentUser = 0
users = [{"name" : "Holger", "score" : 30, "tries" : 1},
         {"name" : "Verena", "score" : 110, "tries" : 4},
         {"name" : "Peter", "score" : 80, "tries" :3}]

print()

def updateScore(newScore):
    users[currentUser]["score"] += newScore

def returnUsers():
    return users

def updateTries():
    users[currentUser]["tries"] += 1

def updateUser(newUser):
    currentUser = newUser

print(returnUsers())
updateScore(300)
print(returnUsers())
updateUser(2)
updateTries()
print(returnUsers())

## Deklarativer / „pure“ Ansatz
statt globale daten zu verändern, geben funktionen neue datenstrukturen zurück.
so wird das verhalten vorhersehbarer und einfacher zu testen.

In [ ]:
# Rekonstruktion: pure Varianten (ohne globale Seiteneffekte)

def update_score(users, current_user, delta):
    """Gibt eine neue User-Liste zurück (Score wird beim aktuellen User erhöht)."""
    new_users = [u.copy() for u in users]
    new_users[current_user]["score"] += delta
    return new_users

def update_tries(users, current_user):
    """Gibt eine neue User-Liste zurück (tries wird beim aktuellen User erhöht)."""
    new_users = [u.copy() for u in users]
    new_users[current_user]["tries"] += 1
    return new_users

def set_current_user(new_user):
    """Pure: gibt einfach den neuen Index zurück."""
    return new_user

def get_users(users):
    """Pure: defensive Kopie (damit Call-Site nichts 'aus Versehen' mutiert)."""
    return [u.copy() for u in users]


# --- Demo wie im Ausgangscode (aber ohne Seiteneffekte) ---
currentUser = 2
users = [
    {"name": "Holger", "score": 30, "tries": 1},
    {"name": "Verena", "score": 110, "tries": 4},
    {"name": "Peter", "score": 80, "tries": 3},
]

print(get_users(users))
users = update_score(users, currentUser, 300)
print(get_users(users))
currentUser = set_current_user(0)
users = update_tries(users, currentUser)
print(get_users(users))


# Closures (Verschachtelte Funktionen)
eine **closure** entsteht, wenn eine innere funktion auf variablen aus dem äusseren scope zugreift und diese „mitnimmt“.

## Beispiel: Multiplikation mit festem Faktor


In [ ]:
# Modul 323-sw2: Beispiel2 Closure
# Verschachtelte Funtionen

def mal(x):
    def mal_x(y):
        return x * y
    return mal_x

malZwei = mal(2)        # malZwei ist ein freeze von mal_x(y) mit x = 2
malDrei = mal(3)        # malDrei ist ein freeze von mal_x(y) mit x = 3

print("5 malZwei gibt: ", malZwei(5))
print("5 malDrei gibt: ", malDrei(5))


## Beispiel: Text auf maximale Länge kürzen
hier wird `max_chars` im äusseren funktionsaufruf „eingefroren“.

In [ ]:
def write(max_chars):

    def write_max(text):

        return text[:max_chars]

    return write_max



name = write(20)



grund = write(100)



print("Dein Name:", name("Alexander Hamilton"))

print("Dein Grund:", grund("Ich will lernen, wie man gute Programme schreibt."))

print("Mein Text:", name("Ich schreibe einen Text, der viel länger ist, als er darf!"))



## Beispiel: Begrüssungen als Closure
mit dem gleichen muster können wir funktionen konfigurieren, z.B. verschiedene begrüssungstexte.

In [ ]:
def gruss(begruessungstext):
    def sage(name):
        return f"{begruessungstext}, {name}!"
    return sage

hallo = gruss("Hallo")
guten_tag = gruss("Guten Tag")

print(hallo("Abel"))
print(guten_tag("Herr Sonderegger"))

## Beispiel: Addierer-Funktionen
ein klassisches closure-beispiel: `addiere(a)` erzeugt eine funktion, die immer **`a` + b** rechnet.

In [ ]:
# Modul 323-sw2: Beispiel2 Closure
# Verschachtelte Funktionen

def addiere(a):
    def addiere_a(b):
        return a + b
    return addiere_a

addFünf = addiere(5)
addZehn = addiere(10)

print(addFünf(7))
print(addZehn(7))

## Beispiel: Kommentar + Uhrzeit formatieren
auch hier wird der kommentartext in der closure gespeichert.

In [ ]:
def comment(kommentar):
    def comment_x(uhrzeit):
        return f"{uhrzeit}: {kommentar}"
    return comment_x

commentOne = comment("Das ist der erste Kommentar")
commentTwo = comment("Das ist der zweite Kommentar")

print(commentOne("13:47"))
print(commentTwo("15:15"))

# Closure + `map()`
`map(f, liste)` wendet eine funktion `f` auf jedes element der liste an.
In kombination mit closures lässt sich `f` sehr elegant konfigurieren.

In [ ]:
# Modul 323-sw2: Beispiel 3 -  Closure mit map()
# Multipliziert eine Liste von Zahlen mit einem Faktor

def mal(x):
   def mal_x(y):
      return x * y
   return mal_x

def mult_liste(liste, faktor):
   mal_f = mal(faktor)
   return list(map(mal_f, liste))

li = [10, 12, 3, 23, 4, 5, 11, 2]
print(mult_liste(li, 10))


## Aufgabe: Adressen mit `map()` erzeugen
aus einem strassennamen und einer liste von hausnummern wird eine liste mit vollständigen adressen erzeugt.

In [ ]:
# Modul 323-sw2: Aufgabe2 Closure mit map()
# Bilde eine Adresse aus einem Strassennamen und einer liste aus Hausnummern

def street(name):                       # outer-function to define the street
    def number(nr):                     # inner-function to add the Number
        return name + " " + str(nr)     # nr has to be casted to string!
    return number

def setAdress(numbers, strt):
	adr = street(strt)
	return list(map(adr, numbers))

houseNumbers = [10, 12, 3, 23, 4, 5, 11, 2]
print(setAdress(houseNumbers, "Bahnhofstrasse"))



# `reduce()` (z.B. Fakultät)
`functools.reduce()` faltet eine sequenz auf einen einzelnen wert.
hier: gakultät, indem fortlaufend multipliziert wird.

In [ ]:
# Modul 323-sw2: Beispiel reduce
from functools import reduce

def mult(x,y):
    print("x=",x," y=",y)
    return x*y

n = 11
fact = reduce(mult, range(1, n))
print('Factorial of {}: {}'.format(n - 1, fact))

# Aufgaben: Closure add
allgemein formuliert mit `outer_add(x)` und `inner_add(y)`.

In [ ]:
# Aufgabe-2.1 Closure add
# Allgemein formuliert mit outer_function und inner_function

def outer_add(x):
    # This inner function is a closure
    def inner_add(y):
        return x + y
    return inner_add

# Create closures with different values of x
closure_add1 = outer_add(10)            # closure_add1 ist ein freez von inner_add(y) mit x = 10!
closure_add2 = outer_add(20)            # closure_add1 ist ein freeze von inner_add(y) mit x = 20!

# Call the closures with different values of y
result1 = closure_add1(5)  # Output: 15 (10 + 5)
result2 = closure_add2(5)  # Output: 25 (20 + 5)

print(result1)
print(result2)


# Aufgabe: Closure-„Adder“ mit internem Zustand
diese aufgabe zeigt das spannungsfeld:
- die funktion ist eine closure und merkt sich werte (liste `data`).
- gleichzeitig ist es nicht mehr pur, weil sich der interne zustand bei jedem aufruf verändert.


In [ ]:
# Write your code here :-)
def make_sum():

    data = []

    def summer(val):

        data.append(val)
        _sum = sum(data)

        return _sum

    print(data)
    return summer



adder = make_sum()
adder2 = make_sum()

print(adder(2))
print(adder(2))
print(adder(2))
print(adder2(5))
print(adder2(2))
print(adder2(2))


---
## Zusammenfassung
- **Abstraktion**: wiederverwendbare funktionen/variablen statt copy-paste.
- **Pure Functions**: Gleicher Input → gleicher output, keine versteckten seiteneffekte.
- **Closures**: "konfigurierte funktionen" durch eingefrorene äussere variablen.
- **`map()`/`reduce()`**: deklarative werkzeuge, um listen zu transformieren oder zu einem wert zu reduzieren.
